# Model Distillation for Computer Vision
## Part 1: Setup & Core Concepts

**Workshop Duration:** 2 hours  
**Level:** Intermediate

---

### Learning Objectives
By the end of this notebook, you will:
1. Understand the model distillation paradigm shift
2. Know how CLIP and Vision Transformers enable zero-shot detection
3. Have a working environment for the hands-on exercises

---

## 1. Environment Setup

Let's verify our environment and install the required packages.

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Install required packages (run once)
# Uncomment if needed - using uv for faster installs

# !uv pip install -q ultralytics>=8.4.0  # For YOLO26 support
# !uv pip install -q autodistill autodistill-grounding-dino autodistill-yolov8
# !uv pip install -q transformers pillow matplotlib supervision

In [ ]:
# Verify installations
import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")

# Should be >= 8.4.0 for YOLO26 support
assert tuple(map(int, ultralytics.__version__.split('.')[:2])) >= (8, 4), \
    "Please upgrade: uv pip install ultralytics>=8.4.0"

---

## 2. The Paradigm Shift: Why Model Distillation?

### Traditional CV Workflow (Old Way)
```
Collect Images → Manual Annotation ($500-$10,000) → Train Model → Deploy
                       ↑
              Time: Weeks to months
```

### Modern Distillation Workflow (New Way)
```
Collect Images → Foundation Model Auto-Labels → Train Lightweight Model → Deploy
                       ↑
              Time: Under 2 hours
              Cost: $50-$200 compute
```

### The Key Insight

> "Manual annotation is itself distillation from the most capable base model available—the human brain. Foundation models can now replace humans for many annotation tasks."

Foundation models like CLIP, SAM, and Grounding DINO were trained on **billions** of images. They possess vast visual knowledge but are:
- Too slow for real-time inference (~30ms+ on high-end GPUs)
- Too large for edge deployment (hundreds of MBs to GBs)
- Too expensive to run at scale

**Solution:** Transfer their knowledge to small, fast models like YOLO26.

### Numbers to Remember

| Metric | Value |
|--------|-------|
| CLIP training data | 400M image-text pairs |
| Zero-shot ImageNet accuracy | 76.2% |
| Achievable accuracy (no annotation) | 70-85% |
| Time to first model | Under 2 hours |
| Recommended images per class | 500-1000 |

---

## 3. Understanding CLIP: The Foundation of Zero-Shot Detection

### What is CLIP?

**CLIP** (Contrastive Language-Image Pretraining) is a model that learns to connect images and text in a shared embedding space.

```
Image → [Image Encoder] → 512-dim Embedding
                                    ↓
                             Cosine Similarity
                                    ↑
Text  → [Text Encoder]  → 512-dim Embedding
```

### How It Works

1. **Dual Encoders:** Separate neural networks for images (ViT or ResNet) and text (Transformer)
2. **Shared Space:** Both outputs are projected into the same 512-dimensional space
3. **Contrastive Learning:** Training maximizes similarity for matching pairs, minimizes for non-matches

In [ ]:
# Let's see CLIP in action for zero-shot classification
from transformers import pipeline
from PIL import Image
import requests
from io import BytesIO

# Load CLIP zero-shot classifier
classifier = pipeline(
    "zero-shot-image-classification",
    model="openai/clip-vit-base-patch32"
)

# Load a sample image
image_url = "http://images.cocodataset.org/val2017/000000039769.jpg"
response = requests.get(image_url)
image = Image.open(BytesIO(response.content))

# Display the image
display(image)

In [ ]:
# Zero-shot classification with ANY text labels
candidate_labels = [
    "a photo of cats on a couch",
    "a photo of dogs playing",
    "a photo of a car",
    "a photo of a person"
]

results = classifier(image, candidate_labels=candidate_labels)

print("Zero-Shot Classification Results:")
print("-" * 40)
for result in results:
    print(f"{result['label']}: {result['score']:.3f}")

### Key Insight

CLIP can classify images into **any category** describable in natural language—without ever being trained on that specific class!

This is the foundation of:
- **Grounding DINO:** Text-prompted object detection
- **SAM 3:** Text-prompted segmentation
- **Autodistill:** Auto-labeling with text prompts

---

## 4. Vision Transformers (ViT): Images as Sequences

### The Key Insight

> "An Image is Worth 16×16 Words"

Vision Transformers treat images as sequences of patches, similar to how language models treat sentences as sequences of words.

### Patch Embedding Process

```
224×224 Image
    ↓
Split into 16×16 patches (196 patches total)
    ↓
Flatten each patch (16×16×3 = 768 dimensions)
    ↓
Add positional embeddings
    ↓
Process through Transformer layers
    ↓
Output embeddings for classification/detection
```

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualize how ViT sees an image as patches
def visualize_patches(image, patch_size=16):
    """Show how an image is divided into patches for ViT processing."""
    # Resize to 224x224 (standard ViT input size)
    img_resized = image.resize((224, 224))
    img_array = np.array(img_resized)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Original image
    axes[0].imshow(img_array)
    axes[0].set_title("Original Image (224×224)")
    axes[0].axis('off')
    
    # Image with patch grid
    axes[1].imshow(img_array)
    for i in range(0, 224, patch_size):
        axes[1].axhline(y=i, color='red', linewidth=0.5)
        axes[1].axvline(x=i, color='red', linewidth=0.5)
    axes[1].set_title(f"ViT Patches ({224//patch_size}×{224//patch_size} = {(224//patch_size)**2} patches)")
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"Each 16×16 patch becomes a 'token' in the sequence")
    print(f"Total sequence length: {(224//patch_size)**2} + 1 (CLS token) = {(224//patch_size)**2 + 1}")

visualize_patches(image)

### Why Transformers Work for Images

| Aspect | CNN | Vision Transformer |
|--------|-----|-------------------|
| Receptive Field | Local, grows gradually | Global from layer 1 |
| Data Requirements | Works with small data | Needs large datasets |
| Transfer Learning | Good | Excellent |
| Multi-modal | Difficult | Natural (same architecture as text) |

**Key Advantage:** Attention allows the model to relate any patch to any other patch immediately, enabling better understanding of global context.

---

## 5. The Autodistill Pipeline

### Core Workflow

```
┌─────────────────┐    ┌──────────────┐    ┌─────────────────┐
│ Unlabeled Images│ → │ Base Model   │ → │ Auto-labeled    │
│                 │    │ + Ontology   │    │ Dataset         │
└─────────────────┘    └──────────────┘    └─────────────────┘
                                                   │
                                                   ▼
┌─────────────────┐    ┌──────────────┐    ┌─────────────────┐
│ Deployed Model  │ ← │ Target Model │ ← │ Training        │
│ (Edge-ready)    │    │ Training     │    │ (YOLO26)        │
└─────────────────┘    └──────────────┘    └─────────────────┘
```

### Key Components

| Component | Purpose | Examples |
|-----------|---------|----------|
| **Base Model** | Large model for auto-labeling | Grounding DINO, GroundedSAM, CLIP |
| **Ontology** | Maps text prompts to class names | `{"milk bottle": "bottle"}` |
| **Target Model** | Small model to train | YOLO26, YOLOv8, RF-DETR |
| **Distilled Model** | Final deployable weights | Edge-ready for specific task |

In [ ]:
# Example: Defining an ontology for auto-labeling
from autodistill.detection import CaptionOntology

# The ontology maps text prompts (what to look for) to class names (how to label)
ontology = CaptionOntology({
    "cat": "cat",
    "remote control": "remote",
    "couch": "couch"
})

print("Ontology defined:")
print(f"  Prompts: {list(ontology.prompts())}")
print(f"  Classes: {list(ontology.classes())}")

### Prompt Engineering Tips

Good prompts are **visually descriptive**:

| Good Prompt | Bad Prompt | Why |
|-------------|------------|-----|
| "person wearing hard hat" | "worker" | Visual vs. abstract |
| "white milk bottle" | "container" | Specific vs. generic |
| "red fire extinguisher" | "safety equipment" | Color + object |
| "cat sitting on couch" | "pet" | Action + context |

---

## 6. YOLO26: The Latest Target Model

**YOLO26** (released January 14, 2026) is the latest evolution of the YOLO series, optimized for edge deployment.

### Key Improvements over Previous Versions

1. **End-to-End NMS-Free:** No post-processing needed, faster inference
2. **43% Faster CPU Inference:** Optimized for edge devices
3. **DFL Removal:** Simplified export to more hardware platforms
4. **MuSGD Optimizer:** Better training stability

### Model Variants

| Model | Parameters | mAP@50-95 | T4 TensorRT (ms) |
|-------|------------|-----------|------------------|
| YOLO26n | 2.4M | 40.9 | 1.7 |
| YOLO26s | 9.5M | 48.6 | 2.5 |
| YOLO26m | 20.4M | 53.1 | 4.7 |
| YOLO26l | 24.8M | 55.0 | 6.2 |
| YOLO26x | 55.7M | 57.5 | 11.8 |

In [ ]:
# Quick demo: YOLO26 inference
from ultralytics import YOLO

# Load YOLO26 nano model (smallest, fastest)
model = YOLO("yolo26n.pt")

# Run inference on our sample image
results = model(image)

# Display results
print(f"Model: YOLO26n")
print(f"Detected {len(results[0].boxes)} objects")
for box in results[0].boxes:
    cls_id = int(box.cls[0])
    conf = float(box.conf[0])
    cls_name = model.names[cls_id]
    print(f"  - {cls_name}: {conf:.2f}")

In [ ]:
# Visualize detections
import matplotlib.pyplot as plt

# Plot results
annotated = results[0].plot()
plt.figure(figsize=(10, 8))
plt.imshow(annotated)
plt.axis('off')
plt.title('YOLO26n Detection Results')
plt.show()

---

## 7. Checkpoint: Knowledge Check

Before we proceed to hands-on exercises, let's verify understanding:

### Quiz

1. **What is the main advantage of model distillation?**
   - [ ] A) Makes models larger
   - [x] B) Transfers knowledge from large models to small, fast models
   - [ ] C) Requires more training data

2. **How does CLIP enable zero-shot classification?**
   - [ ] A) By training on all possible classes
   - [x] B) By mapping images and text to a shared embedding space
   - [ ] C) By using a larger dataset

3. **What is an ontology in Autodistill?**
   - [ ] A) A type of neural network
   - [x] B) A mapping from text prompts to class names
   - [ ] C) A training algorithm

4. **Why use YOLO26 as a target model?**
   - [x] A) Fast inference, small size, edge-deployable
   - [ ] B) Highest accuracy regardless of speed
   - [ ] C) Works without GPU

---

## 8. Summary

### What We Learned

1. **Model distillation** transfers knowledge from large foundation models to small, deployable models
2. **CLIP** enables zero-shot detection by connecting images and text in a shared space
3. **Vision Transformers** treat images as sequences of patches
4. **Autodistill** provides a pipeline: Base Model → Auto-labels → Target Model → Deployed Model
5. **YOLO26** is the latest target model, optimized for edge deployment

### Next Steps

In **Notebook 2**, we'll:
- Use Grounding DINO to auto-label images with text prompts
- Design effective ontologies
- Generate a training dataset from unlabeled images

---

**Continue to:** [02_autolabeling_with_foundation_models.ipynb](./02_autolabeling_with_foundation_models.ipynb)